# Clip a remote TreeMap COG and summarize by county

This notebook reads a Cloud Optimized GeoTIFF (COG) from a direct URL or a STAC Item URL, clips it by an arbitrary vector file, and summarizes raster values by polygon. The default example summarizes a TreeMap-like raster by county for Southeast U.S. states.

Default example COG:

`https://data-to-science-opendata-bucket.s3.us-east-2.amazonaws.com/d2s/ps2.d2s.org/projects/f8934dd1-75c0-4fa0-9c75-b29c8e4c7dc1/flights/b1e45635-3c77-464b-9b1f-414109fb123b/data_products/fbb9d28b-6eb6-4cab-97ae-7dbc50ad84a6/6e7abd64-2c5a-4407-8b8e-7ed0e485e491.tif`

Notes:

- The URL above is a direct COG asset. If you instead pass a STAC Item JSON URL, `resolve_cog_href()` will choose a `.tif` asset.
- The default vector source is the Census generalized county boundary ZIP. Set `VECTOR_PATH` to any local/remote vector readable by GeoPandas (GeoPackage, GeoJSON, Shapefile ZIP, etc.).
- Two summary modes are available:
  - `continuous`: one row per polygon with count, sum, mean, standard deviation, min, max, and quartiles. This is the default because the example COG is `float32`.
  - `categorical_counts`: one row per polygon × raster value with pixel counts and area estimates. Use this for integer rasters such as USFS TreeMap `TM_ID`/`VALUE` rasters.
- For full Southeast runs, remote COG reads can take a while. Start with `MAX_FEATURES_FOR_DEMO = 5` if you want a quick smoke test.

In [1]:
from __future__ import annotations

from dataclasses import dataclass
from pathlib import Path
from typing import Sequence
import json

import geopandas as gpd
import numpy as np
import pandas as pd
import requests
import rasterio
from rasterio.features import geometry_mask
from rasterio.mask import mask
from rasterio.windows import Window
from rasterio.windows import WindowError
from rasterio.features import geometry_window
from shapely.geometry import box, mapping
from shapely.ops import unary_union

## Parameters

Edit these values before running the rest of the notebook.

In [2]:
COG_OR_STAC_URL = "https://data-to-science-opendata-bucket.s3.us-east-2.amazonaws.com/d2s/ps2.d2s.org/projects/f8934dd1-75c0-4fa0-9c75-b29c8e4c7dc1/flights/b1e45635-3c77-464b-9b1f-414109fb123b/data_products/fbb9d28b-6eb6-4cab-97ae-7dbc50ad84a6/6e7abd64-2c5a-4407-8b8e-7ed0e485e491.tif"

# Use None to download Census counties and filter to SOUTHEAST_STATE_FIPS.
# Or set to a local/remote vector file such as "data/raw/my_counties.gpkg".
VECTOR_PATH = None
VECTOR_LAYER = None

# "continuous" for float/metric rasters; "categorical_counts" for integer class/TM_ID rasters.
SUMMARY_MODE = "continuous"

# Census state FIPS codes. Edit this list if your definition of "Southeast" differs.
SOUTHEAST_STATE_FIPS = {
    "01": "Alabama",
    "05": "Arkansas",
    "12": "Florida",
    "13": "Georgia",
    "21": "Kentucky",
    "22": "Louisiana",
    "28": "Mississippi",
    "37": "North Carolina",
    "45": "South Carolina",
    "47": "Tennessee",
    "51": "Virginia",
}

COUNTY_BOUNDARIES_URL = "https://www2.census.gov/geo/tiger/GENZ2023/shp/cb_2023_us_county_500k.zip"

# Works whether the notebook is launched from the repository root or from notebooks/.
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
OUTPUT_DIR = PROJECT_ROOT / "data" / "interim" / "treemap_county_summary"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Set to an integer such as 5 for a fast notebook smoke test. Use None for all polygons.
MAX_FEATURES_FOR_DEMO = None

# Set to True if you also want a clipped COG for the dissolved vector footprint.
WRITE_CLIPPED_RASTER = False
CLIPPED_RASTER_PATH = OUTPUT_DIR / "treemap_southeast_clip.tif"

COUNTY_STATS_CSV_PATH = OUTPUT_DIR / "treemap_southeast_county_stats.csv"
COUNTY_VALUE_COUNTS_CSV_PATH = OUTPUT_DIR / "treemap_southeast_county_value_counts.csv"
STATE_SUMMARY_CSV_PATH = OUTPUT_DIR / "treemap_southeast_state_summary.csv"

## Helpers

In [3]:
GDAL_COG_ENV = {
    "AWS_NO_SIGN_REQUEST": "YES",
    "GDAL_DISABLE_READDIR_ON_OPEN": "EMPTY_DIR",
    "CPL_VSIL_CURL_ALLOWED_EXTENSIONS": ".tif,.TIF,.tiff,.TIFF",
}


def resolve_cog_href(url: str, asset_key: str | None = None) -> str:
    """Return a direct COG href from either a COG URL or a STAC Item URL."""
    lowered = url.lower().split("?", 1)[0]
    if lowered.endswith((".tif", ".tiff")):
        return url

    response = requests.get(url, timeout=30)
    response.raise_for_status()
    item = response.json()
    assets = item.get("assets", {})
    if not assets:
        raise ValueError("URL did not look like a COG and STAC Item has no assets")

    if asset_key is not None:
        try:
            return assets[asset_key]["href"]
        except KeyError as exc:
            raise KeyError(f"Asset key {asset_key!r} not found in STAC Item") from exc

    candidates = []
    for key, asset in assets.items():
        href = asset.get("href", "")
        media_type = asset.get("type", "")
        roles = set(asset.get("roles", []))
        if href.lower().split("?", 1)[0].endswith((".tif", ".tiff")) or "geotiff" in media_type.lower():
            score = 0
            score += 2 if "data" in roles else 0
            score += 1 if key.lower() in {"data", "cog", "image", "raster", "treemap"} else 0
            candidates.append((score, key, href))

    if not candidates:
        raise ValueError("No GeoTIFF-like asset found in STAC Item")
    candidates.sort(reverse=True)
    return candidates[0][2]


def load_vector(vector_path: str | Path | None, layer: str | None = None) -> gpd.GeoDataFrame:
    """Load a user vector or the default Southeast county polygons."""
    if vector_path is not None:
        gdf = gpd.read_file(vector_path, layer=layer)
        if gdf.empty:
            raise ValueError(f"Vector file has no features: {vector_path}")
        if gdf.crs is None:
            raise ValueError("Vector CRS is missing. Define it before clipping/summarizing.")
        return gdf

    counties = gpd.read_file(COUNTY_BOUNDARIES_URL)
    counties = counties[counties["STATEFP"].isin(SOUTHEAST_STATE_FIPS)].copy()
    counties["STATE_NAME"] = counties["STATEFP"].map(SOUTHEAST_STATE_FIPS)
    counties = counties.sort_values(["STATEFP", "COUNTYFP"]).reset_index(drop=True)
    return counties


def inspect_raster(raster_href: str) -> dict:
    """Open the raster and return key metadata."""
    with rasterio.Env(**GDAL_COG_ENV), rasterio.open(raster_href) as src:
        return {
            "driver": src.driver,
            "width": src.width,
            "height": src.height,
            "count": src.count,
            "dtype": src.dtypes[0],
            "crs": str(src.crs),
            "transform": tuple(src.transform),
            "bounds": tuple(src.bounds),
            "nodata": src.nodata,
            "is_tiled": src.is_tiled,
            "block_shapes": src.block_shapes,
        }


def pixel_area(src: rasterio.io.DatasetReader) -> tuple[float | None, str]:
    """Return per-pixel area and units when the raster CRS is projected."""
    if src.crs and src.crs.is_projected:
        return abs(src.transform.a * src.transform.e), "square CRS units"
    return None, "unknown; raster CRS is not projected"

In [4]:
def clipped_bounds_filter(polygons: gpd.GeoDataFrame, src: rasterio.io.DatasetReader) -> gpd.GeoDataFrame:
    """Keep only polygons whose bounding boxes intersect the raster bounds."""
    raster_bounds = gpd.GeoSeries([box(*src.bounds)], crs=src.crs)
    polygons_in_raster_crs = polygons.to_crs(src.crs)
    keep = polygons_in_raster_crs.intersects(raster_bounds.iloc[0])
    return polygons.loc[keep.to_numpy()].copy()


def clip_raster_to_vector(
    raster_href: str,
    polygons: gpd.GeoDataFrame,
    out_path: str | Path,
    *,
    dissolve: bool = True,
) -> Path:
    """Clip a raster to the vector footprint and write a compressed tiled GeoTIFF."""
    out_path = Path(out_path)
    out_path.parent.mkdir(parents=True, exist_ok=True)

    with rasterio.Env(**GDAL_COG_ENV), rasterio.open(raster_href) as src:
        polygons = polygons.to_crs(src.crs)
        geometries = [mapping(unary_union(polygons.geometry))] if dissolve else [mapping(g) for g in polygons.geometry]
        image, transform = mask(src, geometries, crop=True, filled=True, nodata=src.nodata)
        profile = src.profile.copy()
        profile.update(
            driver="GTiff",
            height=image.shape[1],
            width=image.shape[2],
            transform=transform,
            compress="deflate",
            predictor=2,
            tiled=True,
            blockxsize=512,
            blockysize=512,
            bigtiff="IF_SAFER",
        )
        with rasterio.open(out_path, "w", **profile) as dst:
            dst.write(image)

    return out_path

In [5]:
@dataclass(frozen=True)
class ZonalSummaryConfig:
    group_columns: Sequence[str]
    raster_value_column: str = "raster_value"
    all_touched: bool = False
    max_features: int | None = None


def _iter_polygon_pixels(
    raster_href: str,
    polygons: gpd.GeoDataFrame,
    group_columns: Sequence[str],
    *,
    all_touched: bool,
    max_features: int | None,
):
    """Yield feature identifiers, selected pixels, and pixel area for each polygon."""
    missing_columns = [c for c in group_columns if c not in polygons.columns]
    if missing_columns:
        raise KeyError(f"Vector is missing group columns: {missing_columns}")

    with rasterio.Env(**GDAL_COG_ENV), rasterio.open(raster_href) as src:
        working = clipped_bounds_filter(polygons, src)
        if max_features is not None:
            working = working.head(max_features).copy()

        if working.empty:
            raise ValueError("No vector features intersect the raster bounds")

        working = working.to_crs(src.crs)
        area_per_pixel, area_units = pixel_area(src)
        nodata = src.nodata

        for position, (_, feature) in enumerate(working.iterrows(), start=1):
            geom = feature.geometry
            if geom is None or geom.is_empty:
                continue

            try:
                window = geometry_window(src, [mapping(geom)], pad_x=0, pad_y=0)
            except WindowError:
                continue

            # Clamp in case numerical precision nudges the geometry just outside bounds.
            full_window = Window(col_off=0, row_off=0, width=src.width, height=src.height)
            window = window.intersection(full_window)
            if window.width <= 0 or window.height <= 0:
                continue

            values = src.read(1, window=window, masked=False)
            if values.size == 0:
                continue

            transform = src.window_transform(window)
            inside = geometry_mask(
                [mapping(geom)],
                out_shape=values.shape,
                transform=transform,
                invert=True,
                all_touched=all_touched,
            )
            valid = inside.copy()
            if nodata is not None:
                valid &= values != nodata
            if np.issubdtype(values.dtype, np.floating):
                valid &= np.isfinite(values)

            selected = values[valid]
            if selected.size == 0:
                continue

            identifiers = {column: feature[column] for column in group_columns}
            yield identifiers, selected, area_per_pixel, area_units

            if position % 25 == 0:
                print(f"Processed {position:,} / {len(working):,} polygons")


def summarize_continuous_raster_by_polygons(
    raster_href: str,
    polygons: gpd.GeoDataFrame,
    config: ZonalSummaryConfig,
) -> pd.DataFrame:
    """Compute continuous zonal statistics inside each polygon."""
    rows = []
    for identifiers, selected, area_per_pixel, area_units in _iter_polygon_pixels(
        raster_href,
        polygons,
        config.group_columns,
        all_touched=config.all_touched,
        max_features=config.max_features,
    ):
        values = selected.astype("float64", copy=False)
        row = {
            **identifiers,
            "pixel_count": int(values.size),
            "value_sum": float(values.sum()),
            "value_mean": float(values.mean()),
            "value_std": float(values.std(ddof=0)),
            "value_min": float(values.min()),
            "value_p25": float(np.percentile(values, 25)),
            "value_p50": float(np.percentile(values, 50)),
            "value_p75": float(np.percentile(values, 75)),
            "value_max": float(values.max()),
        }
        if area_per_pixel is not None:
            row["area_square_crs_units"] = row["pixel_count"] * area_per_pixel
            row["area_hectares"] = row["area_square_crs_units"] / 10_000
            row["area_acres"] = row["area_square_crs_units"] * 0.00024710538146717
        else:
            row["area_units_note"] = area_units
        rows.append(row)

    return pd.DataFrame(rows)


def summarize_categorical_raster_by_polygons(
    raster_href: str,
    polygons: gpd.GeoDataFrame,
    config: ZonalSummaryConfig,
) -> pd.DataFrame:
    """Count categorical raster values inside each polygon.

    The implementation reads one bounding-box window per polygon, so it works with
    remote COGs without downloading the full raster first.
    """
    rows: list[pd.DataFrame] = []
    for identifiers, selected, area_per_pixel, area_units in _iter_polygon_pixels(
        raster_href,
        polygons,
        config.group_columns,
        all_touched=config.all_touched,
        max_features=config.max_features,
    ):
        unique_values, counts = np.unique(selected, return_counts=True)
        feature_summary = pd.DataFrame(
            {
                config.raster_value_column: unique_values,
                "pixel_count": counts.astype("int64"),
            }
        )
        for column, value in identifiers.items():
            feature_summary[column] = value

        if area_per_pixel is not None:
            feature_summary["area_square_crs_units"] = feature_summary["pixel_count"] * area_per_pixel
            feature_summary["area_hectares"] = feature_summary["area_square_crs_units"] / 10_000
            feature_summary["area_acres"] = feature_summary["area_square_crs_units"] * 0.00024710538146717
        else:
            feature_summary["area_units_note"] = area_units

        rows.append(feature_summary)

    if not rows:
        return pd.DataFrame(columns=[*config.group_columns, config.raster_value_column, "pixel_count"])

    ordered_columns = [*config.group_columns, config.raster_value_column, "pixel_count"]
    area_columns = ["area_square_crs_units", "area_hectares", "area_acres", "area_units_note"]
    summary = pd.concat(rows, ignore_index=True)
    ordered_columns.extend([c for c in area_columns if c in summary.columns])
    remaining_columns = [c for c in summary.columns if c not in ordered_columns]
    return summary[ordered_columns + remaining_columns]

## Load the COG and vector features

In [6]:
raster_href = resolve_cog_href(COG_OR_STAC_URL)
raster_info = inspect_raster(raster_href)
print(json.dumps(raster_info, indent=2, default=str))

{
  "driver": "GTiff",
  "width": 154179,
  "height": 97279,
  "count": 1,
  "dtype": "float32",
  "crs": "EPSG:5070",
  "transform": [
    30.0,
    0.0,
    -2361585.0,
    0.0,
    -30.0,
    3177435.0000000037,
    0.0,
    0.0,
    1.0
  ],
  "bounds": [
    -2361585.0,
    259065.00000000373,
    2263785.0,
    3177435.0000000037
  ],
  "nodata": 3.4028234663852886e+38,
  "is_tiled": true,
  "block_shapes": [
    [
      512,
      512
    ]
  ]
}


In [7]:
polygons = load_vector(VECTOR_PATH, layer=VECTOR_LAYER)
print(f"Loaded {len(polygons):,} vector features")
print(polygons.crs)
polygons.head()

Loaded 1,008 vector features
EPSG:4269


,STATEFP,COUNTYFP,COUNTYNS,GEOIDFQ,GEOID,NAME,NAMELSAD,STUSPS,STATE_NAME,LSAD,ALAND,AWATER,geometry
0,01,001,00161526,0500000US01001,01001,Autauga,Autauga County,AL,Alabama,06,1539631461,25677536,"POLYGON ((-86.9212 32.65754, -86.92035 32.6585..."
1,01,003,00161527,0500000US01003,01003,Baldwin,Baldwin County,AL,Alabama,06,4117725048,1132887203,"POLYGON ((-88.02858 30.22676, -88.02399 30.230..."
2,01,005,00161528,0500000US01005,01005,Barbour,Barbour County,AL,Alabama,06,2292160151,50523213,"POLYGON ((-85.74803 31.61918, -85.74544 31.618..."
3,01,007,00161529,0500000US01007,01007,Bibb,Bibb County,AL,Alabama,06,1612188713,9572302,"POLYGON ((-87.42194 33.00338, -87.31854 33.006..."
4,01,009,00161530,0500000US01009,01009,Blount,Blount County,AL,Alabama,06,1670259100,14860281,"POLYGON ((-86.96336 33.85822, -86.95967 33.857..."


## Optional: write a clipped raster

This clips the COG to the dissolved vector footprint. Leave `WRITE_CLIPPED_RASTER = False` if you only need zonal summaries.

In [8]:
if WRITE_CLIPPED_RASTER:
    clipped_path = clip_raster_to_vector(raster_href, polygons, CLIPPED_RASTER_PATH)
    print(f"Wrote {clipped_path}")
else:
    print("Skipping clipped raster write; set WRITE_CLIPPED_RASTER = True to enable.")

Skipping clipped raster write; set WRITE_CLIPPED_RASTER = True to enable.


## Summarize raster values by county

For the Census county default, the grouping columns are state/county identifiers plus names. If you use your own vector, replace `group_columns` with columns that uniquely identify each polygon.

- Use `SUMMARY_MODE = "continuous"` for the example `float32` COG and other metric rasters.
- Use `SUMMARY_MODE = "categorical_counts"` for class rasters or USFS TreeMap `TM_ID`/`VALUE` rasters.

In [ ]:
if VECTOR_PATH is None:
    group_columns = ["STATEFP", "STATE_NAME", "COUNTYFP", "NAME", "GEOID"]
else:
    # Change this for custom vector files. If unsure, use the first non-geometry column.
    candidate_columns = [c for c in polygons.columns if c != polygons.geometry.name]
    group_columns = candidate_columns[:1]
    print(f"Using group_columns={group_columns}; edit this cell for richer identifiers.")

summary_config = ZonalSummaryConfig(
    group_columns=group_columns,
    raster_value_column="treemap_value",
    all_touched=False,
    max_features=MAX_FEATURES_FOR_DEMO,
)

if SUMMARY_MODE == "continuous":
    county_summary = summarize_continuous_raster_by_polygons(raster_href, polygons, summary_config)
    county_summary.to_csv(COUNTY_STATS_CSV_PATH, index=False)
    print(f"Wrote {len(county_summary):,} county statistic rows to {COUNTY_STATS_CSV_PATH}")
elif SUMMARY_MODE == "categorical_counts":
    county_summary = summarize_categorical_raster_by_polygons(raster_href, polygons, summary_config)
    county_summary.to_csv(COUNTY_VALUE_COUNTS_CSV_PATH, index=False)
    print(f"Wrote {len(county_summary):,} county × raster-value rows to {COUNTY_VALUE_COUNTS_CSV_PATH}")
else:
    raise ValueError('SUMMARY_MODE must be "continuous" or "categorical_counts"')

county_summary.head()

Processed 25 / 1,008 polygons
Processed 50 / 1,008 polygons
Processed 75 / 1,008 polygons
Processed 100 / 1,008 polygons


In [ ]:
# County-level coverage QA. In continuous mode this is already one row per county.
if SUMMARY_MODE == "continuous":
    county_coverage = county_summary[[*group_columns, "pixel_count", "area_hectares", "area_acres"]].copy()
else:
    aggregations = {
        "unique_raster_values": ("treemap_value", "nunique"),
        "pixel_count": ("pixel_count", "sum"),
    }
    if "area_hectares" in county_summary.columns:
        aggregations["area_hectares"] = ("area_hectares", "sum")
        aggregations["area_acres"] = ("area_acres", "sum")
    county_coverage = county_summary.groupby(group_columns, dropna=False).agg(**aggregations).reset_index()

county_coverage_path = OUTPUT_DIR / "treemap_southeast_county_coverage.csv"
county_coverage.to_csv(county_coverage_path, index=False)
print(f"Wrote {len(county_coverage):,} county coverage rows to {county_coverage_path}")
county_coverage.head()

## Optional: summarize at state level

In [ ]:
if {"STATEFP", "STATE_NAME"}.issubset(county_summary.columns):
    if SUMMARY_MODE == "continuous":
        state_summary = (
            county_summary
            .groupby(["STATEFP", "STATE_NAME"], dropna=False)
            .agg(
                pixel_count=("pixel_count", "sum"),
                value_sum=("value_sum", "sum"),
                value_min=("value_min", "min"),
                value_max=("value_max", "max"),
                area_hectares=("area_hectares", "sum") if "area_hectares" in county_summary.columns else ("pixel_count", "sum"),
                area_acres=("area_acres", "sum") if "area_acres" in county_summary.columns else ("pixel_count", "sum"),
            )
            .reset_index()
        )
        state_summary["value_mean"] = state_summary["value_sum"] / state_summary["pixel_count"]
    else:
        state_summary = (
            county_summary
            .groupby(["STATEFP", "STATE_NAME", "treemap_value"], dropna=False)
            .agg(
                pixel_count=("pixel_count", "sum"),
                area_hectares=("area_hectares", "sum") if "area_hectares" in county_summary.columns else ("pixel_count", "sum"),
                area_acres=("area_acres", "sum") if "area_acres" in county_summary.columns else ("pixel_count", "sum"),
            )
            .reset_index()
        )

    state_summary.to_csv(STATE_SUMMARY_CSV_PATH, index=False)
    print(f"Wrote {len(state_summary):,} state summary rows to {STATE_SUMMARY_CSV_PATH}")
    display(state_summary.head())
else:
    print("State columns not available for this vector file.")

## Joining TreeMap attributes later

This notebook intentionally keeps the raster summary generic. If `treemap_value` corresponds to a TreeMap `TM_ID`/`VALUE`, join the output CSV to the TreeMap value attribute table (VAT) or a lookup table to attach attributes such as `PLT_CN`, forest type, basal area, TPA, or carbon. Keep FIA control numbers as strings when possible to avoid precision loss.